# **DuckDB** Views

## Iceburg Table Connection

### Load Jar Files

In [ ]:
import os
import sys
# 1. Set PYSPARK_SUBMIT_ARGS to match your working batch file launcher
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    '--conf spark.driver.extraClassPath="C:/data/spark/jars/iceberg-spark-runtime-4.0_2.13-1.10.0.jar" '
    "pyspark-shell"
)

# 2. Ensure Python paths align for the worker processes
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

### Spark Connection

In [ ]:
from pyspark.sql import functions as sf
from pyspark.sql import window as sw
from pyspark.sql import types as sdt
from pyspark.sql import SparkSession
from datetime import datetime

LOCAL_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_warehouse"
STG_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_staging_warehouse"
RPT_WAREHOUSE_PATH = "/data/data_files/iceberg/WideWorldImportersDW"

MSSQL_JAR = r"C:/data/spark/jars/mssql-jdbc-12.6.5.jre11.jar"

CATALOG_NAME = "local"
STG_CATALOG_NAME = "staging"
WH_CATALOG_NAME = "reporting"

# staging_table_name = "staging.Integration.employee_Staging"
# wh_table_name = "reporting.dimension.Employees"

spark = SparkSession.builder \
    .appName("Iceberg Setup") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", f"file:///{LOCAL_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.warehouse", f"file:///{STG_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.warehouse", f"file:///{RPT_WAREHOUSE_PATH}") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .getOrCreate()


spark.catalog.setCurrentCatalog(WH_CATALOG_NAME)

spark
# spark.sql("SHOW CATALOGS").show(truncate=False)
# spark.sql("SHOW NAMESPACES IN reporting").show(truncate=False)
# spark.sql("SHOW DATABASES IN reporting").show(truncate=False)
# spark.sql("SHOW TABLES IN reporting.dimension").show(truncate=False)
for ns in spark.sql("SHOW NAMESPACES IN reporting").collect():
    namespace = ns["namespace"]
    # print(f"\nNamespace: {namespace}")
    spark.sql(f"SHOW TABLES IN reporting.{namespace}").show(truncate=False)


## DuckDB Connection

In [ ]:
# import sqlite3
# import pandas as pd
import duckdb

# SQLITE_DB_PATH = r"C:\Users\progr\Downloads\WideWorldImporters.db"

con = duckdb.connect(r"C:\data\my_warehouse.duckdb", read_only=True)

TestSQL = """
SELECT 
city.WWI_City_ID AS "City Code",
city.City AS "City Name",
city.State_Province AS "State/Province",
city.Country AS "Country",
city.Continent AS "Continent",
city.Sales_Territory AS "Sales Territory",
city.Region AS "Region",
city.Subregion AS "Subregion",
city.Latest_Recorded_Population AS "Latest Recorded Population",
customer.WWI_Customer_ID AS "Customer Code",
customer.Customer AS "Customer Name",
customer.Bill_To_Customer AS "Billing Customer Name",
customer.Category AS "Customer Category",
customer.Buying_Group AS "Buying Customer Group",
customer.Primary_Contact AS "Primary Contact",
customer.Postal_Code AS "Postal Code",
stockitem.WWI_Stock_Item_ID AS "Stock Item Code",
stockitem.Stock_Item AS "Stock Item Name",
stockitem.Color AS "Item Color",
stockitem.Selling_Package AS "Item Selling Package",
stockitem.Buying_Package AS "Item Buying Package",
stockitem.Brand AS "ItemBrand",
stockitem.Size As "Item Size",
stockitem.Lead_Time_Days AS "Item Lead Time (Days)",
stockitem.Quantity_Per_Outer AS "Item Quantity Per Outer",
stockitem.Is_Chiller_Stock AS "Is Chiller Item Stock",
stockitem.Barcode AS "Item Barcode",
stockitem.Tax_Rate AS "Item Tax Rate",
stockitem.Unit_Price AS "Item Unit Price",
stockitem.Recommended_Retail_Price AS "Item Recommended Retail Price",
stockitem.Typical_Weight_Per_Unit AS "Item Typical Weight Per Unit",
orderDate.ISO_Week_Number AS "Order ISO Week Number",
orderDate.Date AS "Order Date",
orderDate.Day_Number AS "Order Day Number",
orderDate.Day AS "Order Day",
orderDate.Month AS "Order Month",
orderDate.Short_Month AS "Order Short Month",
orderDate.Calendar_Month_Number AS "Order Calendar Month Number",
orderDate.Calendar_Month_Label AS "Order Calendar Month Label",
orderDate.Calendar_Year AS "Order Calendar Year",
orderDate.Calendar_Year_Label AS "Order Calendar Year Label",
orderDate.Fiscal_Month_Number AS "Order Fiscal Month Number",
orderDate.Fiscal_Month_Label AS "Order Fiscal Month Label",
orderDate.Fiscal_Year AS "Order Fiscal Year",
orderDate.Fiscal_Year_Label AS "Order Fiscal Year Label",
pickerDate.Date AS "Picker Date",
pickerDate.Day_Number AS "Picker Day Number",
pickerDate.Day AS "Picker Day",
pickerDate.Month AS "Picker Month",
pickerDate.Short_Month AS "Picker Short Month",
pickerDate.Calendar_Month_Number AS "Picker Calendar Month Number",
pickerDate.Calendar_Month_Label AS "Picker Calendar Month Label",
pickerDate.Calendar_Year AS "Picker Calendar Year",
pickerDate.Calendar_Year_Label AS "Picker Calendar Year Label",
pickerDate.Fiscal_Month_Number AS "Picker Fiscal Month Number",
pickerDate.Fiscal_Month_Label AS "Picker Fiscal Month Label",
pickerDate.Fiscal_Year AS "Picker Fiscal Year",
pickerDate.Fiscal_Year_Label AS "Picker Fiscal Year Label",
pickerDate.ISO_Week_Number AS "Picker ISO Week Number",
SalesPerson.WWI_Employee_ID AS "Sales Person Code",
SalesPerson.Employee AS "Sales Person Name",
SalesPerson.Preferred_Name AS "Sales Person Preferred Name",
SalesPerson.Is_Salesperson AS "Is Sales Person",
PickerPerson.WWI_Employee_ID AS "Picker Person Code",
PickerPerson.Employee AS "Picker Person Name",
PickerPerson.Preferred_Name AS "Picker Person Preferred Name",
PickerPerson.Is_Salesperson AS "Is Picker Person",
orders.WWI_Order_ID AS "Order Code",
orders.WWI_Backorder_ID AS "Order Backorder Code",
orders.Description AS "Order Description",
orders.Package AS "Order Package",
orders.Quantity AS "Order Quantity",
orders.Unit_Price AS "Order Unit Price",
orders.Tax_Rate AS "Order Tax Rate",
orders.Total_Excluding_Tax AS "Order Total Excluding Tax",
orders.Tax_Amount AS "Order Tax Amount",
orders.Total_Including_Tax AS "Order Total Including Tax"
FROM 
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/fact/order', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) AS orders LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/City', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN)))  AS city ON orders.City_Key = city.City_Key LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/Customer', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN)))  AS customer ON orders.Customer_Key = customer.Customer_Key LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/Stock_Item', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN)))  AS stockitem ON orders.Stock_Item_Key = stockitem.Stock_Item_Key LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/Date', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) AS orderDate ON orders.Order_Date_Key = orderDate.Date LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/Date', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) AS pickerDate ON orders.Picked_Date_Key = pickerDate.Date LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/Employee', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN)))  AS SalesPerson ON orders.Salesperson_Key = SalesPerson.Employee_Key LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/Employee', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN)))  AS PickerPerson ON orders.Salesperson_Key = PickerPerson.Employee_Key AND orders.Picker_Key = PickerPerson.Employee_Key
LIMIT 10
"""

df_objects = con.execute(TestSQL).df()

display(df_objects)

# with sqlite3.connect(SQLITE_DB_PATH) as conn:
#     # 1. Read table into a Pandas DataFrame
#     df_orders = pd.read_sql_query(TestSQL, conn)
#     print(df_orders.head())
con.close()
    

## Load DataFrame

In [ ]:
df_purchase = spark.table("reporting.Fact.Purchase").alias("Purchase")
df_stock_holding = spark.table("reporting.Fact.Stock_Holding").alias("Stock_Holding")
df_order = spark.table("reporting.Fact.Order").alias("Order")
df_movement = spark.table("reporting.Fact.Movement").alias("Movement")
df_sale = spark.table("reporting.Fact.Sale").alias("Sale")
df_payment_method = spark.table("reporting.Dimension.payment_method").alias("payment_method")
df_supplier = spark.table("reporting.Dimension.supplier").alias("supplier")
df_city = spark.table("reporting.Dimension.city").alias("city")
df_stock_item = spark.table("reporting.Dimension.stock_item").alias("stock_item")
df_customer = spark.table("reporting.Dimension.customer").alias("customer")
df_date = spark.table("reporting.Dimension.date").alias("date")
df_transaction_type = spark.table("reporting.Dimension.transaction_type").alias("transaction_type")
df_employee = spark.table("reporting.Dimension.employee").alias("employee")
df_transaction = spark.table("reporting.Fact.Transaction").alias("Transaction")

### Fact df_order Analysis

In [ ]:
thesql = """
SELECT 
city.WWI_City_ID AS `City Code`,
city.City AS `City Name`,
city.State_Province AS `State/Province`,
city.Country AS `Country`,
city.Continent AS `Continent`,
city.Sales_Territory AS `Sales Territory`,
city.Region AS `Region`,
city.Subregion AS `Subregion`,
city.Latest_Recorded_Population AS `Latest Recorded Population`,
customer.WWI_Customer_ID AS `Customer Code`,
customer.Customer AS `Customer Name`,
customer.Bill_To_Customer AS `Billing Customer Name`,
customer.Category AS `Customer Category`,
customer.Buying_Group AS `Buying Customer Group`,
customer.Primary_Contact AS `Primary Contact`,
customer.Postal_Code AS `Postal Code`,
stockitem.WWI_Stock_Item_ID AS `Stock Item Code`,
stockitem.Stock_Item AS `Stock Item Name`,
stockitem.Color AS `Item Color`,
stockitem.Selling_Package AS `Item Selling Package`,
stockitem.Buying_Package AS `Item Buying Package`,
stockitem.Brand AS `ItemBrand`,
stockitem.Size As `Item Size`,
stockitem.Lead_Time_Days AS `Item Lead Time (Days)`,
stockitem.Quantity_Per_Outer AS `Item Quantity Per Outer`,
stockitem.Is_Chiller_Stock AS `Is Chiller Item Stock`,
stockitem.Barcode AS `Item Barcode`,
stockitem.Tax_Rate AS `Item Tax Rate`,
stockitem.Unit_Price AS `Item Unit Price`,
stockitem.Recommended_Retail_Price AS `Item Recommended Retail Price`,
stockitem.Typical_Weight_Per_Unit AS `Item Typical Weight Per Unit`,
orderDate.ISO_Week_Number AS `Order ISO Week Number`,
orderDate.Date AS `Order Date`,
orderDate.Day_Number AS `Order Day Number`,
orderDate.Day AS `Order Day`,
orderDate.Month AS `Order Month`,
orderDate.Short_Month AS `Order Short Month`,
orderDate.Calendar_Month_Number AS `Order Calendar Month Number`,
orderDate.Calendar_Month_Label AS `Order Calendar Month Label`,
orderDate.Calendar_Year AS `Order Calendar Year`,
orderDate.Calendar_Year_Label AS `Order Calendar Year Label`,
orderDate.Fiscal_Month_Number AS `Order Fiscal Month Number`,
orderDate.Fiscal_Month_Label AS `Order Fiscal Month Label`,
orderDate.Fiscal_Year AS `Order Fiscal Year`,
orderDate.Fiscal_Year_Label AS `Order Fiscal Year Label`,
pickerDate.Date AS `Picker Date`,
pickerDate.Day_Number AS `Picker Day Number`,
pickerDate.Day AS `Picker Day`,
pickerDate.Month AS `Picker Month`,
pickerDate.Short_Month AS `Picker Short Month`,
pickerDate.Calendar_Month_Number AS `Picker Calendar Month Number`,
pickerDate.Calendar_Month_Label AS `Picker Calendar Month Label`,
pickerDate.Calendar_Year AS `Picker Calendar Year`,
pickerDate.Calendar_Year_Label AS `Picker Calendar Year Label`,
pickerDate.Fiscal_Month_Number AS `Picker Fiscal Month Number`,
pickerDate.Fiscal_Month_Label AS `Picker Fiscal Month Label`,
pickerDate.Fiscal_Year AS `Picker Fiscal Year`,
pickerDate.Fiscal_Year_Label AS `Picker Fiscal Year Label`,
pickerDate.ISO_Week_Number AS `Picker ISO Week Number`,
SalesPerson.WWI_Employee_ID AS `Sales Person Code`,
SalesPerson.Employee AS `Sales Person Name`,
SalesPerson.Preferred_Name AS `Sales Person Preferred Name`,
SalesPerson.Is_Salesperson AS `Is Sales Person`,
PickerPerson.WWI_Employee_ID AS `Picker Person Code`,
PickerPerson.Employee AS `Picker Person Name`,
PickerPerson.Preferred_Name AS `Picker Person Preferred Name`,
PickerPerson.Is_Salesperson AS `Is Picker Person`,
orders.WWI_Order_ID AS `Order Code`,
orders.WWI_Backorder_ID AS `Order Backorder Code`,
orders.Description AS `Order Description`,
orders.Package AS `Order Package`,
orders.Quantity AS `Order Quantity`,
orders.Unit_Price AS `Order Unit Price`,
orders.Tax_Rate AS `Order Tax Rate`,
orders.Total_Excluding_Tax AS `Order Total Excluding Tax`,
orders.Tax_Amount AS `Order Tax Amount`,
orders.Total_Including_Tax AS `Order Total Including Tax`
FROM Fact.Order AS orders INNER JOIN
Dimension.City AS city ON orders.City_Key = city.City_Key INNER JOIN
Dimension.Customer AS customer ON orders.Customer_Key = customer.Customer_Key INNER JOIN
Dimension.Stock_Item AS stockitem ON orders.Stock_Item_Key = stockitem.Stock_Item_Key INNER JOIN
Dimension.Date AS orderDate ON orders.Order_Date_Key = orderDate.Date INNER JOIN
Dimension.Date AS pickerDate ON orders.Picked_Date_Key = pickerDate.Date INNER JOIN
Dimension.Employee AS SalesPerson ON orders.Salesperson_Key = SalesPerson.Employee_Key INNER JOIN
Dimension.Employee AS PickerPerson ON orders.Picker_Key = PickerPerson.Employee_Key
"""
spark.sql(thesql).show()

In [ ]:
df_order.printSchema()
df_order.filter(df_order.Picker_Key > 0).orderBy("Order_Date_Key").show(10)

In [ ]:
df_employee.filter((df_employee.Employee_Key == 11) | (df_employee.Employee_Key == 20)).show(10)

In [ ]:
df_customer.filter((df_customer.Customer_Key == 11) | (df_customer.Customer_Key == 20)).show(10, truncate=False)

In [ ]:
df_date.show(5, truncate=False)

## DuckDB Connection

### Create View SQL

#### customer_order_details

```sql
SELECT 
city.WWI_City_ID AS "City Code",
city.City AS "City Name", city.State_Province AS "State/Province", city.Country AS "Country", city.Continent AS "Continent", city.Sales_Territory AS "Sales Territory", city.Region AS "Region", city.Subregion AS "Subregion", city.Latest_Recorded_Population AS "Latest Recorded Population", 
customer.WWI_Customer_ID AS "Customer Code", customer.Customer AS "Customer Name", customer.Bill_To_Customer AS "Billing Customer Name", customer.Category AS "Customer Category", customer.Buying_Group AS "Buying Customer Group", customer.Primary_Contact AS "Primary Contact", customer.Postal_Code AS "Postal Code", stockitem.WWI_Stock_Item_ID AS "Stock Item Code", stockitem.Stock_Item AS "Stock Item Name", stockitem.Color AS "Item Color", stockitem.Selling_Package AS "Item Selling Package", stockitem.Buying_Package AS "Item Buying Package", stockitem.Brand AS "ItemBrand", stockitem.Size As "Item Size", stockitem.Lead_Time_Days AS "Item Lead Time (Days)", stockitem.Quantity_Per_Outer AS "Item Quantity Per Outer", stockitem.Is_Chiller_Stock AS "Is Chiller Item Stock", stockitem.Barcode AS "Item Barcode", stockitem.Tax_Rate AS "Item Tax Rate", stockitem.Unit_Price AS "Item Unit Price", stockitem.Recommended_Retail_Price AS "Item Recommended Retail Price", stockitem.Typical_Weight_Per_Unit AS "Item Typical Weight Per Unit",  orderDate.ISO_Week_Number AS "Order ISO Week Number", orderDate.Date AS "Order Date", orderDate.Day_Number AS "Order Day Number", orderDate.Day AS "Order Day", orderDate.Month AS "Order Month", orderDate.Short_Month AS "Order Short Month", orderDate.Calendar_Month_Number AS "Order Calendar Month Number", orderDate.Calendar_Month_Label AS "Order Calendar Month Label", orderDate.Calendar_Year AS "Order Calendar Year", orderDate.Calendar_Year_Label AS "Order Calendar Year Label", orderDate.Fiscal_Month_Number AS "Order Fiscal Month Number", orderDate.Fiscal_Month_Label AS "Order Fiscal Month Label", orderDate.Fiscal_Year AS "Order Fiscal Year", orderDate.Fiscal_Year_Label AS "Order Fiscal Year Label",
pickerDate.Date AS "Picker Date", pickerDate.Day_Number AS "Picker Day Number", pickerDate.Day AS "Picker Day", pickerDate.Month AS "Picker Month", pickerDate.Short_Month AS "Picker Short Month", pickerDate.Calendar_Month_Number AS "Picker Calendar Month Number", pickerDate.Calendar_Month_Label AS "Picker Calendar Month Label", pickerDate.Calendar_Year AS "Picker Calendar Year", pickerDate.Calendar_Year_Label AS "Picker Calendar Year Label", pickerDate.Fiscal_Month_Number AS "Picker Fiscal Month Number", pickerDate.Fiscal_Month_Label AS "Picker Fiscal Month Label", pickerDate.Fiscal_Year AS "Picker Fiscal Year", pickerDate.Fiscal_Year_Label AS "Picker Fiscal Year Label", pickerDate.ISO_Week_Number AS "Picker ISO Week Number", 
SalesPerson.WWI_Employee_ID AS "Sales Person Code", SalesPerson.Employee AS "Sales Person Name", SalesPerson.Preferred_Name AS "Sales Person Preferred Name", SalesPerson.Is_Salesperson AS "Is Sales Person" PickerPerson.WWI_Employee_ID AS "Picker Person Code", PickerPerson.Employee AS "Picker Person Name", PickerPerson.Preferred_Name AS "Picker Person Preferred Name", PickerPerson.Is_Salesperson AS "Is Picker Person", 
orders.WWI_Order_ID AS "Order Code", orders.WWI_Backorder_ID AS "Order Backorder Code", orders.Description AS "Order Description", orders.Package AS "Order Package", orders.Quantity AS "Order Quantity", orders.Unit_Price AS "Order Unit Price", orders.Tax_Rate AS "Order Tax Rate", orders.Total_Excluding_Tax AS "Order Total Excluding Tax", orders.Tax_Amount AS "Order Tax Amount", orders.Total_Including_Tax AS "Order Total Including Tax"
FROM Fact.Order AS orders INNER JOIN
Dimension.City AS city ON orders.City_Key = city.City_Key INNER JOIN
Dimension.Customer AS customer ON orders.Customer_Key = customer.Customer_Key INNER JOIN
Dimension.Stock_Item AS stockitem ON orders.Stock_Item_Key = stockitem.Stock_Item_Key INNER JOIN
Dimension.Date AS orderDate ON orders.Order_Date_Key = orderDate.Date INNER JOIN
Dimension.Date AS pickerDate ON orders.Picked_Date_Key = pickerDate.Date INNER JOIN
Dimension.Employee AS SalesPerson ON orders.Salesperson_Key = SalesPerson.Employee_Key INNER JOIN
Dimension.Employee AS PickerPerson ON orders.Salesperson_Key = PickerPerson.Employee_Key AND orders.Picker_Key = PickerPerson.Employee_Key
```

In [50]:
import duckdb

factOrderView = """
SELECT 
city.WWI_City_ID AS "City Code",
city.City AS "City Name",
city.State_Province AS "State/Province",
city.Country AS "Country",
city.Continent AS "Continent",
city.Sales_Territory AS "Sales Territory",
city.Region AS "Region",
city.Subregion AS "Subregion",
city.Latest_Recorded_Population AS "Latest Recorded Population",
customer.WWI_Customer_ID AS "Customer Code",
customer.Customer AS "Customer Full Name",
customer.Bill_To_Customer AS "Billing Customer Name",
customer.Category AS "Customer Category",
customer.Buying_Group AS "Buying Customer Group",
customer.Primary_Contact AS "Primary Contact",
customer.Postal_Code AS "Postal Code",
stockitem.WWI_Stock_Item_ID AS "Item ID",
stockitem.Stock_Item AS "Stock Item Name",
stockitem.Color AS "Item Color",
stockitem.Selling_Package AS "Item Selling Package",
stockitem.Buying_Package AS "Item Buying Package",
stockitem.Brand AS "ItemBrand",
stockitem.Size As "Item Size",
stockitem.Lead_Time_Days AS "Item Lead Time (Days)",
stockitem.Quantity_Per_Outer AS "Item Quantity Per Outer",
stockitem.Is_Chiller_Stock AS "Is Chiller Item Stock",
stockitem.Barcode AS "Item Barcode",
stockitem.Tax_Rate AS "Item Tax Rate",
stockitem.Unit_Price AS "Item Unit Price",
stockitem.Recommended_Retail_Price AS "Item Recommended Retail Price",
stockitem.Typical_Weight_Per_Unit AS "Item Typical Weight Per Unit",
SalesPerson.WWI_Employee_ID AS "Sales Person Code",
SalesPerson.Employee AS "Sales Person Name",
SalesPerson.Preferred_Name AS "Sales Person Preferred Name",
SalesPerson.Is_Salesperson AS "Is Sales Person",
PickerPerson.WWI_Employee_ID AS "Picker Person Code",
PickerPerson.Employee AS "Picker Person Name",
PickerPerson.Preferred_Name AS "Picker Person Preferred Name",
PickerPerson.Is_Salesperson AS "Is Picker Person",
orders.Order_Date_Key,
orders.Picked_Date_Key,
orders.WWI_Order_ID AS "Order Code",
orders.WWI_Backorder_ID AS "Order Backorder Code",
orders.Description AS "Order Description",
orders.Package AS "Order Package",
orders.Quantity AS "Order Quantity",
orders.Unit_Price AS "Order Unit Price",
orders.Tax_Rate AS "Order Tax Rate",
orders.Total_Excluding_Tax AS "Order Total Excluding Tax",
orders.Tax_Amount AS "Order Tax Amount",
orders.Total_Including_Tax AS "Order Total Including Tax"
FROM 
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/fact/order', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) AS orders LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/City', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN)))  AS city ON orders.City_Key = city.City_Key LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/Customer', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN)))  AS customer ON orders.Customer_Key = customer.Customer_Key LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/Stock_Item', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN)))  AS stockitem ON orders.Stock_Item_Key = stockitem.Stock_Item_Key LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/Employee', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN)))  AS SalesPerson ON orders.Salesperson_Key = SalesPerson.Employee_Key LEFT OUTER JOIN
iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/Dimension/Employee', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN)))  AS PickerPerson ON orders.Salesperson_Key = PickerPerson.Employee_Key AND orders.Picker_Key = PickerPerson.Employee_Key
"""

# Open database in read-write mode
con = duckdb.connect(r"C:\data\my_warehouse.duckdb", read_only=False)

# Make sure the Iceberg extension is loaded
con.execute("INSTALL iceberg; LOAD iceberg;")

CreateViewSQL = f"""
CREATE OR REPLACE VIEW customer_order_details AS
{factOrderView}
"""

con.execute(CreateViewSQL)
print("✅ View 'customer_order_details' created permanently in my_warehouse.duckdb!")
con.close()

✅ View 'customer_order_details' created permanently in my_warehouse.duckdb!


#### Purchases_Invoice

```sql
SELECT  city.WWI_City_ID AS "City Code", city.City AS "City Name", city.State_Province AS "State/Province", city.Country AS "Country", city.Continent AS "Continent", city.Sales_Territory AS "Sales Territory", city.Region AS "Region", city.Subregion AS "Subregion", city.Latest_Recorded_Population AS "Latest 
Recorded Population",customer.WWI_Customer_ID AS "Customer Code",customer.Customer AS "Customer Name",customer.Bill_To_Customer AS "Billing Customer Name",customer.Category AS "Customer Category",customer.Buying_Group AS "Buying Customer Group",customer.Primary_Contact AS "Primary Contact",customer.Postal_Code AS "Postal Code", 
stockitem.WWI_Stock_Item_ID AS "Item ID", stockitem.Stock_Item AS "Stock Item Name", stockitem.Color AS "Item Color", stockitem.Selling_Package AS "Item Selling Package", stockitem.Buying_Package AS "Item Buying Package", stockitem.Brand AS "ItemBrand", stockitem.Size As "Item Size", stockitem.Lead_Time_Days AS "Item Lead Time (Days)", stockitem.Quantity_Per_Outer AS "Item Quantity Per Outer", stockitem.Is_Chiller_Stock AS "Is Chiller Item Stock", stockitem.Barcode AS "Item Barcode", stockitem.Tax_Rate AS "Item Tax Rate", stockitem.Unit_Price AS "Item Unit Price", stockitem.Recommended_Retail_Price AS "Item Recommended Retail Price", stockitem.Typical_Weight_Per_Unit AS "Item Typical Weight Per Unit",
orderDate.ISO_Week_Number AS "Order ISO Week Number",orderDate.Date AS "Order Date",orderDate.Day_Number AS "Order Day Number",orderDate.Day AS "Order Day",orderDate.Month AS "Order Month",orderDate.Short_Month AS "Order Short Month",orderDate.Calendar_Month_Number AS "Order Calendar Month Number",orderDate.Calendar_Month_Label AS "Order Calendar Month Label",orderDate.Calendar_Year AS "Order Calendar Year",orderDate.Calendar_Year_Label AS "Order Calendar Year Label",orderDate.Fiscal_Month_Number AS "Order Fiscal Month Number",orderDate.Fiscal_Month_Label AS "Order Fiscal Month Label",orderDate.Fiscal_Year AS "Order Fiscal Year",orderDate.Fiscal_Year_Label AS "Order Fiscal Year Label",
pickerDate.Date AS "Picker Date", pickerDate.Day_Number AS "Picker Day Number", pickerDate.Day AS "Picker Day", pickerDate.Month AS "Picker Month", pickerDate.Short_Month AS "Picker Short Month", pickerDate.Calendar_Month_Number AS "Picker Calendar Month Number", pickerDate.Calendar_Month_Label AS "Picker Calendar Month Label", pickerDate.Calendar_Year AS "Picker Calendar Year", pickerDate.Calendar_Year_Label AS "Picker Calendar Year Label", pickerDate.Fiscal_Month_Number AS "Picker Fiscal Month Number", pickerDate.Fiscal_Month_Label AS "Picker Fiscal Month Label", pickerDate.Fiscal_Year AS "Picker Fiscal Year", pickerDate.Fiscal_Year_Label AS "Picker Fiscal Year Label", pickerDate.ISO_Week_Number AS "Picker ISO Week Number",
SalesPerson.WWI_Employee_ID AS "Sales Person Code", SalesPerson.Employee AS "Sales Person Name", SalesPerson.Preferred_Name AS "Sales Person Preferred Name", SalesPerson.Is_Salesperson AS "Is Sales Person",
PickerPerson.WWI_Employee_ID AS "Picker Person Code", PickerPerson.Employee AS "Picker Person Name", PickerPerson.Preferred_Name AS "Picker Person Preferred Name", PickerPerson.Is_Salesperson AS "Is Picker Person",
orders.WWI_Order_ID AS "Order Code", orders.WWI_Backorder_ID AS "Order Backorder Code", orders.Description AS "Order Description", orders.Package AS "Order Package", orders.Quantity AS "Order Quantity", orders.Unit_Price AS "Order Unit Price", orders.Tax_Rate AS "Order Tax Rate", orders.Total_Excluding_Tax AS "Order Total Excluding Tax", orders.Tax_Amount AS "Order Tax Amount", orders.Total_Including_Tax AS "Order Total Including Tax"
FROM 
orders LEFT OUTER JOIN
city ON orders.City_Key = city.City_Key LEFT OUTER JOIN
customer ON orders.Customer_Key = customer.Customer_Key LEFT OUTER JOIN
stockitem ON orders.Stock_Item_Key = stockitem.Stock_Item_Key LEFT OUTER JOIN
orderDate ON orders.Order_Date_Key = orderDate.Date LEFT OUTER JOIN
pickerDate ON orders.Picked_Date_Key = pickerDate.Date LEFT OUTER JOIN
SalesPerson ON orders.Salesperson_Key = SalesPerson.Employee_Key LEFT OUTER JOIN
PickerPerson ON orders.Salesperson_Key = PickerPerson.Employee_Key AND orders.Picker_Key = PickerPerson.Employee_Key
```

In [51]:
import duckdb

factPurchaseView = """
SELECT
    purchase.WWI_Purchase_Order_ID AS "Purchase Code",
    supplier.WWI_Supplier_ID AS "Supplier Code",
    supplier.Supplier AS "Supplier Name",
    supplier.Category AS "Supplier Category",
    supplier.Primary_Contact AS "Primary Contact",
    supplier.Supplier_Reference AS "Supplier Reference",
    supplier.Payment_Days AS "Payment Days",
    supplier.Postal_Code AS "Postal Code",
    Stock_Item.WWI_Stock_Item_ID AS "Item Code",
    Stock_Item.Stock_Item AS "Item Name",
    Stock_Item.Color AS "Item Color",
    Stock_Item.Selling_Package AS "Item Selling Package",
    Stock_Item.Buying_Package AS "Item Buying Package",
    Stock_Item.Brand AS "Item Brand",
    Stock_Item.Size AS "Item Size",
    Stock_Item.Lead_Time_Days AS "ItemLead Time Days",
    Stock_Item.Quantity_Per_Outer AS "Item Quantity Per Outer",
    Stock_Item.Is_Chiller_Stock AS "Is Chiller Item",
    Stock_Item.Barcode AS "Item Barcode",
    Stock_Item.Tax_Rate AS "Item Tax Rate",
    Stock_Item.Unit_Price AS "Item Unit Price",
    Stock_Item.Recommended_Retail_Price AS "Item Recommended Retail Price",
    Stock_Item.Typical_Weight_Per_Unit AS "Item Typical Weight Per Unit",
    purchase.Date_Key As purchase_date_Key,
    purchase.Ordered_Outers AS "Ordered Outers",
    purchase.Ordered_Quantity AS "Ordered Quantity",
    purchase.Received_Outers AS "Received Outers",
    purchase.Package AS "Purchases Package",
    purchase.Is_Order_Finalized AS "Is Purchases Finalized"
FROM
    iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/fact/purchase', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) AS purchase
LEFT JOIN iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/dimension/date', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) AS purchase_date ON
    ((purchase.Date_Key = purchase_date.Date))
LEFT JOIN iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/dimension/supplier', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) AS supplier ON
    ((purchase.Supplier_Key = supplier.Supplier_Key))
LEFT JOIN iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/dimension/Stock_Item', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) AS Stock_Item ON
    ((purchase.Stock_Item_Key = Stock_Item.Stock_Item_Key));
"""

# Open database in read-write mode
con = duckdb.connect(r"C:\data\my_warehouse.duckdb", read_only=False)

# Make sure the Iceberg extension is loaded
con.execute("INSTALL iceberg; LOAD iceberg;")

CreateViewSQL = f"""
CREATE OR REPLACE VIEW Purchases_Invoice AS
{factOrderView}
"""

con.execute(CreateViewSQL)
print("✅ View 'Purchases_Invoice' created permanently in my_warehouse.duckdb!")
con.close()

✅ View 'Purchases_Invoice' created permanently in my_warehouse.duckdb!


#### Sale_Invoice

```sql
SELECT
    factSale.WWI_Invoice_ID AS "Invoice Code",
    dimension_city.WWI_City_ID AS "City Code", dimension_city.City AS "City Name", dimension_city.State_Province AS "State or Province", dimension_city.Country AS Country, dimension_city.Continent AS Continent, dimension_city.Sales_Territory AS "Sales Territory", dimension_city.Region AS Region, dimension_city.Subregion AS Subregion, dimension_city.Latest_Recorded_Population AS "Latest Recorded Population in City",
    customers.WWI_Customer_ID AS "Customer Code", customers.Customer AS "Customer Full Name", customers.Bill_To_Customer AS "Billing Customer Full Name", customers.Category AS "Customer Category", customers.Buying_Group AS "Customer Buying Group", customers.Primary_Contact AS "Customer Primary Contact", customers.Postal_Code AS "Customer Postal Code",
    cbill.WWI_Customer_ID AS "Billing Customer Code", cbill.Customer AS "Billing Customer Full Name", cbill.Bill_To_Customer AS "Billing Customer Full Name", cbill.Category AS "Billing Customer Category", cbill.Buying_Group AS "Billing Customer Buying Group", cbill.Primary_Contact AS "Billing Customer Primary Contact", cbill.Postal_Code AS "Billing Customer Postal Code",
    Stock_Item.WWI_Stock_Item_ID AS "Item Code", Stock_Item.Stock_Item AS "Item Name", Stock_Item.Color AS "Item Color", Stock_Item.Selling_Package AS "Item Selling Package", Stock_Item.Buying_Package AS "Item Buying Package", Stock_Item.Brand AS "Item Brand", Stock_Item.Size AS "Item Size", Stock_Item.Lead_Time_Days AS "ItemLead Time Days", Stock_Item.Quantity_Per_Outer AS "Item Quantity Per Outer", Stock_Item.Is_Chiller_Stock AS "Is Chiller Item", Stock_Item.Barcode AS "Item Barcode", Stock_Item.Tax_Rate AS "Item Tax Rate", Stock_Item.Unit_Price AS "Item Unit Price", Stock_Item.Recommended_Retail_Price AS "Item Recommended Retail Price", Stock_Item.Typical_Weight_Per_Unit AS "Item Typical Weight Per Unit",
    inv_date.Date AS "Invoice Dates", inv_date.Day_Number AS "Invoice Day Number", inv_date."Day" AS "Invoice Day", inv_date."Month" AS "Invoice Month Name", inv_date.Short_Month AS "Invoice Short Month Name", inv_date.Calendar_Month_Number AS "Invoice Month Number", inv_date.Calendar_Month_Label AS "Invoice Calendar Month Name", inv_date.Calendar_Year AS "Invoice Year Number", inv_date.Calendar_Year_Label AS "Invoice Calendar Year Name", inv_date.Fiscal_Month_Number AS "Invoice Financial Month Number", inv_date.Fiscal_Month_Label AS "Invoice Financial Month Name", inv_date.Fiscal_Year AS "Invoice Financial Year", inv_date.Fiscal_Year_Label AS "Invoice Financial Year Name", inv_date.ISO_Week_Number AS "Invoice Week Number",
    del_date.Date AS "Delivery Dates", del_date.Day_Number AS "Delivery Day Number", del_date."Day" AS "Delivery Day", del_date."Month" AS "Delivery Month Name", del_date.Short_Month AS "Delivery Short Month Name", del_date.Calendar_Month_Number AS "Delivery Month Number", del_date.Calendar_Month_Label AS "Delivery Calendar Month Name", del_date.Calendar_Year AS "Delivery Year Number", del_date.Calendar_Year_Label AS "Delivery Calendar Year Name", del_date.Fiscal_Month_Number AS "Delivery Financial Month Number", del_date.Fiscal_Month_Label AS "Delivery Financial Month Name", del_date.Fiscal_Year AS "Delivery Financial Year", del_date.Fiscal_Year_Label AS "Delivery Financial Year Name", del_date.ISO_Week_Number AS "Delivery Week Number",
    employee.Employee AS "Employee Full Name", employee.Preferred_Name AS "Employee Calling Name",
    factSale.Description AS "Billing Note Description / Annotation", factSale.Package AS "Packaging Type", factSale.Quantity AS "Sale Quantity", factSale.Unit_Price AS "Unit Price of Item", factSale.Tax_Rate AS "Tax Rate on Item", factSale.Total_Excluding_Tax AS "Total Excluding Tax on Invoice", factSale.Tax_Amount AS "Tax Amount on Item", factSale.Profit AS "Earned Profit on Invoice", factSale.Total_Including_Tax AS "Tax Amount on Invoice", factSale.Total_Dry_Items AS "Total Item can be stored in Dry Place", factSale.Total_Chiller_Items AS "Total Item need stored in cooler or child Place"
FROM    factSale LEFT JOIN 
        customers ON (factSale.Customer_Key = customers.Customer_Key) LEFT JOIN 
        cbill ON (factSale.Bill_To_Customer_Key = cbill.Customer_Key) LEFT JOIN
        dimension_city ON (factSale.city_key = dimension_city.city_key) LEFT JOIN 
        Stock_Item ON (factSale.Stock_Item_Key = Stock_Item.Stock_Item_Key) LEFT JOIN 
        employee ON (factSale.Salesperson_key = employee.Employee_Key) LEFT JOIN 
        inv_date ON (factSale.Invoice_Date_Key = inv_date.Date) LEFT JOIN 
        del_date ON (factSale.Delivery_Date_Key = del_date.Date);
```

In [49]:
import duckdb

factSaleView = """
SELECT
    factSale.WWI_Invoice_ID AS "Invoice Code",
    dimension_city.WWI_City_ID AS "City Code",
    dimension_city.City AS "City Name",
    dimension_city.State_Province AS "State or Province",
    dimension_city.Country AS Country,
    dimension_city.Continent AS Continent,
    dimension_city.Sales_Territory AS "Sales Territory",
    dimension_city.Region AS Region,
    dimension_city.Subregion AS Subregion,
    dimension_city.Latest_Recorded_Population AS "Latest Recorded Population in City",
    customers.WWI_Customer_ID AS "Customer Code",
    customers.Customer AS "Customer Full Name",
    customers.Bill_To_Customer AS "Billing Customer Full Name",
    customers.Category AS "Customer Category",
    customers.Buying_Group AS "Customer Buying Group",
    customers.Primary_Contact AS "Customer Primary Contact",
    customers.Postal_Code AS "Customer Postal Code",
    cbill.WWI_Customer_ID AS "Billing Customer Code",
    cbill.Customer AS "Billing Customer Full Name",
    cbill.Bill_To_Customer AS "Billing Customer Full Name",
    cbill.Category AS "Billing Customer Category",
    cbill.Buying_Group AS "Billing Customer Buying Group",
    cbill.Primary_Contact AS "Billing Customer Primary Contact",
    cbill.Postal_Code AS "Billing Customer Postal Code",
    Stock_Item.WWI_Stock_Item_ID AS "Item Code",
    Stock_Item.Stock_Item AS "Item Name",
    Stock_Item.Color AS "Item Color",
    Stock_Item.Selling_Package AS "Item Selling Package",
    Stock_Item.Buying_Package AS "Item Buying Package",
    Stock_Item.Brand AS "Item Brand",
    Stock_Item.Size AS "Item Size",
    Stock_Item.Lead_Time_Days AS "ItemLead Time Days",
    Stock_Item.Quantity_Per_Outer AS "Item Quantity Per Outer",
    Stock_Item.Is_Chiller_Stock AS "Is Chiller Item",
    Stock_Item.Barcode AS "Item Barcode",
    Stock_Item.Tax_Rate AS "Item Tax Rate",
    Stock_Item.Unit_Price AS "Item Unit Price",
    Stock_Item.Recommended_Retail_Price AS "Item Recommended Retail Price",
    Stock_Item.Typical_Weight_Per_Unit AS "Item Typical Weight Per Unit",
    employee.Employee AS "Employee Full Name",
    employee.Preferred_Name AS "Employee Calling Name",
    factSale.Invoice_Date_Key,
    factSale.Delivery_Date_Key,
    factSale.Description AS "Billing Note Description / Annotation",
    factSale.Package AS "Packaging Type",
    factSale.Quantity AS "Sale Quantity",
    factSale.Unit_Price AS "Unit Price of Item",
    factSale.Tax_Rate AS "Tax Rate on Item",
    factSale.Total_Excluding_Tax AS "Total Excluding Tax on Invoice",
    factSale.Tax_Amount AS "Tax Amount on Item",
    factSale.Profit AS "Earned Profit on Invoice",
    factSale.Total_Including_Tax AS "Tax Amount on Invoice",
    factSale.Total_Dry_Items AS "Total Item can be stored in Dry Place",
    factSale.Total_Chiller_Items AS "Total Item need stored in cooler or child Place"
FROM
    iceberg_scan('file:///C:/data/data_files/iceberg/WideWorldImportersDW/fact/sale', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) AS factSale
LEFT JOIN iceberg_scan ('file:///C:/data/data_files/iceberg/WideWorldImportersDW/dimension/customer',("version" = '?'),(allow_moved_paths = CAST('t' AS BOOLEAN))) AS customers 
ON (factSale.Customer_Key = customers.Customer_Key)
LEFT JOIN iceberg_scan ('file:///C:/data/data_files/iceberg/WideWorldImportersDW/dimension/customer', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) AS cbill 
ON (factSale.Bill_To_Customer_Key = cbill.Customer_Key)
LEFT JOIN iceberg_scan ('file:///C:/data/data_files/iceberg/WideWorldImportersDW/dimension/city', ("version" = '?'), (allow_moved_paths = CAST ('t' AS BOOLEAN))) AS dimension_city 
ON (factSale.city_key = dimension_city.city_key)
LEFT JOIN iceberg_scan ('file:///C:/data/data_files/iceberg/WideWorldImportersDW/dimension/Stock_Item', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) AS Stock_Item 
ON (factSale.Stock_Item_Key = Stock_Item.Stock_Item_Key)
LEFT JOIN iceberg_scan ('file:///C:/data/data_files/iceberg/WideWorldImportersDW/dimension/employee', ("version" = '?'), (allow_moved_paths = CAST ('t' AS BOOLEAN))) AS employee 
ON (factSale.Salesperson_key = employee.Employee_Key);
"""

# Open database in read-write mode
con = duckdb.connect(r"C:\data\my_warehouse.duckdb", read_only=False)

# Make sure the Iceberg extension is loaded
con.execute("INSTALL iceberg; LOAD iceberg;")

CreateViewSQL = f"""
CREATE OR REPLACE VIEW Sales_Invoice AS
{factSaleView}
"""

con.execute(CreateViewSQL)
print("✅ View 'Sales_Invoice' created permanently in my_warehouse.duckdb!")
con.close()

✅ View 'Sales_Invoice' created permanently in my_warehouse.duckdb!


#### Date Dim

In [72]:
import duckdb

date_dimension_view = """
SELECT
date_key AS "Date Key",
year AS "year",
year_short AS "year in two digits",
month AS "month number",
month_name AS "month name",
month_name_short AS "month name (short)",
day AS "day",
formatted_date AS "days in characters",
day_name AS "weekday name",
day_name_short AS "weekday name (short)",
quarter AS "quarter of year",
week_of_year AS "week number of year",
day_of_week AS "week number",
day_of_year AS "day number of year",
is_weekend AS "weekend or not",
british_date_format AS "british date format (dd/mm/yyyy)",
american_date_format AS "american date format (mm/dd/yyyy)",
iso_date_format AS "iso date format (yyyy-mm-dd)",
yesterday_date AS "yesterday date",
today_date AS "today date",
tomorrow_date AS "tomorrow date",
first_day_of_month AS "first day of month",
last_day_of_month AS "last day of month",
first_day_of_year AS "first day of year",
last_day_of_year AS "last day of year"
FROM
iceberg_scan ('file:///C:/data/data_files/iceberg/WideWorldImportersDW/dimension/dates',("version" = '?'),(allow_moved_paths = CAST('t' AS BOOLEAN)))
"""

# Open database in read-write mode
con = duckdb.connect(r"C:\data\my_warehouse.duckdb", read_only=False)

# Make sure the Iceberg extension is loaded
con.execute("INSTALL iceberg; LOAD iceberg;")

CreateViewSQL = f"""
CREATE OR REPLACE VIEW dimension_dates AS
{date_dimension_view}
"""

con.execute(CreateViewSQL)
print("✅ View 'dimension_dates' created permanently in my_warehouse.duckdb!")
con.close()

✅ View 'dimension_dates' created permanently in my_warehouse.duckdb!


### Test Views

In [ ]:
import duckdb

con = duckdb.connect(r"C:\data\my_warehouse.duckdb", read_only=False)

# Fetch view names matching your pattern
matching_views = con.sql("""
    SELECT view_name 
    FROM duckdb_views() 
    WHERE view_name LIKE 'fact_%'
""").fetchall()
matching_views = con.sql(
    "SELECT schema_name, view_name, sql FROM duckdb_views() where view_name in (')"
).fetchall()

if matching_views:
    view_names = [v[0] for v in matching_views]
    
    # Loop over each view name and drop them one by one
    for view_name in view_names:
        con.execute(f'DROP VIEW IF EXISTS "{view_name}"')
        
    print(f"Dropped views: {view_names}")

con.close()

Dropped views: ['main', 'main', 'main', 'main', 'main', 'main', 'information_schema', 'information_schema', 'information_schema', 'information_schema', 'information_schema', 'information_schema', 'information_schema', 'information_schema', 'information_schema', 'information_schema', 'information_schema', 'main', 'main', 'main', 'main', 'main', 'main', 'main', 'main', 'main', 'main', 'main', 'main', 'main', 'main', 'pg_catalog', 'pg_catalog', 'pg_catalog', 'pg_catalog', 'pg_catalog', 'pg_catalog', 'pg_catalog', 'pg_catalog', 'pg_catalog', 'pg_catalog', 'pg_catalog', 'pg_catalog', 'pg_catalog', 'pg_catalog', 'pg_catalog', 'pg_catalog', 'pg_catalog', 'pg_catalog', 'pg_catalog', 'pg_catalog', 'pg_catalog', 'pg_catalog']


In [76]:
import duckdb

con = duckdb.connect(r"C:\data\my_warehouse.duckdb", read_only=True)

# Returns a DataFrame or table with view details
views = con.sql(
    "SELECT schema_name, view_name, sql FROM duckdb_views()"
).df()

display(views)

con.close()

,schema_name,view_name,sql
0,main,customer_order_details,CREATE VIEW customer_order_details AS SELECT c...
1,main,dimension_dates,CREATE VIEW dimension_dates AS SELECT date_key...
2,main,Purchases_Invoice,CREATE VIEW Purchases_Invoice AS SELECT city.W...
3,main,Purchases_Invoices,CREATE VIEW Purchases_Invoices AS SELECT purch...
4,main,Sales_Invoice,CREATE VIEW Sales_Invoice AS SELECT factSale.W...
5,main,Sale_Invoice,CREATE VIEW Sale_Invoice AS SELECT fs.WWI_Invo...
6,information_schema,character_sets,CREATE TEMP VIEW information_schema.character_...
7,information_schema,check_constraints,CREATE TEMP VIEW information_schema.check_cons...
8,information_schema,columns,"CREATE TEMP VIEW information_schema.""columns"" ..."
9,information_schema,constraint_column_usage,CREATE TEMP VIEW information_schema.constraint...


In [74]:
# import sqlite3
# import pandas as pd
import duckdb

# SQLITE_DB_PATH = r"C:\Users\progr\Downloads\WideWorldImporters.db"

con = duckdb.connect(r"C:\data\my_warehouse.duckdb", read_only=True)

ViewSQL = """
SELECT 
*    
FROM customer_order_details
LIMIT 10
"""

df_objects = con.execute(ViewSQL).df()

display(df_objects)

ViewSQL = """
SELECT 
*
FROM Purchases_Invoice
LIMIT 10
"""

df_objects = con.execute(ViewSQL).df()

display(df_objects)

ViewSQL = """
SELECT 
*
FROM Sales_Invoice
LIMIT 10
"""

df_objects = con.execute(ViewSQL).df()

display(df_objects)


   
ViewSQL = """
SELECT 
*
FROM dimension_dates
LIMIT 10
"""

df_objects = con.execute(ViewSQL).df()

display(df_objects)


con.close()
    

,City Code,City Name,State/Province,Country,Continent,Sales Territory,Region,Subregion,Latest Recorded Population,Customer Code,...,Order Code,Order Backorder Code,Order Description,Order Package,Order Quantity,Order Unit Price,Order Tax Rate,Order Total Excluding Tax,Order Tax Amount,Order Total Including Tax
0,9772,East Dailey,West Virginia,United States,North America,Southeast,Americas,Northern America,557,60,...,10595,10683,Tape dispenser (Red),Each,40,32.00,15.0,1280.00,192.00,1472.00
1,29123,Roachtown,Illinois,United States,North America,Great Lakes,Americas,Northern America,0,0,...,55799,55841,Clear packaging tape 48mmx100m,Each,60,3.50,15.0,210.00,31.50,241.50
2,36499,West Elkton,Ohio,United States,North America,Great Lakes,Americas,Northern America,197,0,...,1475,1525,Void fill 100 L bag (White) 100L,Each,80,12.50,15.0,1000.00,150.00,1150.00
3,14041,Greycliff,Montana,United States,North America,Rocky Mountain,Americas,Northern America,112,0,...,63172,63198,10 mm Double sided bubble wrap 20m,Each,10,30.00,15.0,300.00,45.00,345.00
4,6809,Clewiston,Florida,United States,North America,Southeast,Americas,Northern America,7155,93,...,55401,55438,Halloween zombie mask (Light Brown) M,Each,36,18.00,15.0,648.00,97.20,745.20
5,24530,North Beach Haven,New Jersey,United States,North America,Mideast,Americas,Northern America,2235,506,...,45448,45473,Black and yellow heavy despatch tape 48mmx100m,Each,48,4.10,15.0,196.80,29.52,226.32
6,36434,Wesson,Arkansas,United States,North America,Southeast,Americas,Northern America,0,0,...,54005,54039,Superhero action jacket (Blue) XL,Each,8,30.00,15.0,240.00,36.00,276.00
7,16430,Imlaystown,New Jersey,United States,North America,Mideast,Americas,Northern America,0,51,...,64333,64386,"""The Gu"" red shirt XML tag t-shirt (Black) XS",Each,24,18.00,15.0,432.00,64.80,496.80
8,17730,Kinder,Louisiana,United States,North America,Southeast,Americas,Northern America,2477,0,...,44804,44868,Pack of 12 action figures (female),Packet,9,16.00,15.0,144.00,21.60,165.60
9,4491,Buell,Missouri,United States,North America,Plains,Americas,Northern America,0,198,...,44190,44204,Packing knife with metal insert blade (Yellow)...,Each,25,1.89,15.0,47.25,7.09,54.34


,City Code,City Name,State/Province,Country,Continent,Sales Territory,Region,Subregion,Latest Recorded Population,Customer Code,...,Order Code,Order Backorder Code,Order Description,Order Package,Order Quantity,Order Unit Price,Order Tax Rate,Order Total Excluding Tax,Order Tax Amount,Order Total Including Tax
0,38036,Yerington,Nevada,United States,North America,Far West,Americas,Northern America,3048,0,...,65582,65666,Chocolate echidnas 250g,Bag,144,8.55,10.0,1231.2,123.12,1354.32
1,29887,Ruthsburg,Maryland,United States,North America,Mideast,Americas,Northern America,0,601,...,3991,4007,Halloween skull mask (Gray) XL,Each,120,18.00,15.0,2160.0,324.00,2484.00
2,23396,Muir,Michigan,United States,North America,Great Lakes,Americas,Northern America,604,49,...,12516,12545,Red and white urgent heavy despatch tape 48m...,Each,168,4.10,15.0,688.8,103.32,792.12
3,35810,Wanaque,New Jersey,United States,North America,Mideast,Americas,Northern America,11116,0,...,3487,3524,"""The Gu"" red shirt XML tag t-shirt (Black) 6XL",Each,120,18.00,15.0,2160.0,324.00,2484.00
4,28907,Ridgemark,California,United States,North America,Far West,Americas,Northern America,3016,471,...,50716,50758,RC toy sedan car with remote control (Green) 1...,Each,3,25.00,15.0,75.0,11.25,86.25
5,30839,Scofield,Utah,United States,North America,Rocky Mountain,Americas,Northern America,24,117,...,17867,17875,Air cushion machine (Blue),Each,3,1899.00,15.0,5697.0,854.55,6551.55
6,24164,New Zion,South Carolina,United States,North America,Southeast,Americas,Northern America,0,0,...,14919,14955,Air cushion film 200mmx200mm 325m,Each,1,90.00,15.0,90.0,13.50,103.50
7,14041,Greycliff,Montana,United States,North America,Rocky Mountain,Americas,Northern America,112,0,...,64469,64502,Furry gorilla with big eyes slippers (Black) L,Each,10,32.00,15.0,320.0,48.00,368.00
8,17999,Koontzville,Washington,United States,North America,Far West,Americas,Northern America,0,30,...,43020,43085,Furry gorilla with big eyes slippers (Black) L,Each,9,32.00,15.0,288.0,43.20,331.20
9,30906,Sea Island,Georgia,United States,North America,Southeast,Americas,Northern America,0,0,...,44585,44599,Small 9mm replacement blades 9mm,Each,100,4.10,15.0,410.0,61.50,471.50


,Invoice Code,City Code,City Name,State or Province,Country,Continent,Sales Territory,Region,Subregion,Latest Recorded Population in City,...,Packaging Type,Sale Quantity,Unit Price of Item,Tax Rate on Item,Total Excluding Tax on Invoice,Tax Amount on Item,Earned Profit on Invoice,Tax Amount on Invoice,Total Item can be stored in Dry Place,Total Item need stored in cooler or child Place
0,26014,33237,Sunburg,Minnesota,United States,North America,Plains,Americas,Northern America,100,...,Each,120,18.00,15.0,2160.0,324.00,1200.0,2484.00,120,0
1,40165,15783,Hollywood Park,Texas,United States,North America,Southwest,Americas,Northern America,3062,...,Pair,24,5.00,15.0,120.0,18.00,84.0,138.00,24,0
2,70124,3018,Biggs,California,United States,North America,Far West,Americas,Northern America,1707,...,Each,9,35.00,15.0,315.0,47.25,162.0,362.25,9,0
3,61405,17039,Jeromesville,Ohio,United States,North America,Great Lakes,Americas,Northern America,562,...,Each,96,18.00,15.0,1728.0,259.20,1008.0,1987.20,96,0
4,39023,17039,Jeromesville,Ohio,United States,North America,Great Lakes,Americas,Northern America,562,...,Each,50,1.28,15.0,64.0,9.60,34.0,73.60,50,0
5,35865,19434,Lime Lake,New York,United States,North America,Mideast,Americas,Northern America,867,...,Each,72,18.00,15.0,1296.0,194.40,792.0,1490.40,72,0
6,30946,18865,Laurence Harbor,New Jersey,United States,North America,Mideast,Americas,Northern America,6536,...,Each,7,25.00,15.0,175.0,26.25,87.5,201.25,7,0
7,27477,34568,Tumacacori,Arizona,United States,North America,Southwest,Americas,Northern America,393,...,Each,24,4.10,15.0,98.4,14.76,51.6,113.16,24,0
8,53475,21191,Mashulaville,Mississippi,United States,North America,Southeast,Americas,Northern America,0,...,Each,80,105.00,15.0,8400.0,1260.00,4640.0,9660.00,80,0
9,7678,24700,North Ridge,New York,United States,North America,Mideast,Americas,Northern America,0,...,Each,8,32.00,15.0,256.0,38.40,160.0,294.40,8,0


,Date Key,year,year in two digits,month number,month name,month name (short),day,days in characters,weekday name,weekday name (short),...,british date format (dd/mm/yyyy),american date format (mm/dd/yyyy),iso date format (yyyy-mm-dd),yesterday date,today date,tomorrow date,first day of month,last day of month,first day of year,last day of year
0,2023-01-01,2023,23,1,January,Jan,1,1st,Sunday,Sun,...,01/01/2023,01/01/2023,2023-01-01,2022-12-31,2023-01-01,2023-01-02,2023-01-01,2023-01-31,2023-01-01,2023-12-31
1,2023-01-02,2023,23,1,January,Jan,2,2nd,Monday,Mon,...,02/01/2023,01/02/2023,2023-01-02,2023-01-01,2023-01-02,2023-01-03,2023-01-01,2023-01-31,2023-01-01,2023-12-31
2,2023-01-03,2023,23,1,January,Jan,3,3rd,Tuesday,Tue,...,03/01/2023,01/03/2023,2023-01-03,2023-01-02,2023-01-03,2023-01-04,2023-01-01,2023-01-31,2023-01-01,2023-12-31
3,2023-01-04,2023,23,1,January,Jan,4,4th,Wednesday,Wed,...,04/01/2023,01/04/2023,2023-01-04,2023-01-03,2023-01-04,2023-01-05,2023-01-01,2023-01-31,2023-01-01,2023-12-31
4,2023-01-05,2023,23,1,January,Jan,5,5th,Thursday,Thu,...,05/01/2023,01/05/2023,2023-01-05,2023-01-04,2023-01-05,2023-01-06,2023-01-01,2023-01-31,2023-01-01,2023-12-31
5,2023-01-06,2023,23,1,January,Jan,6,6th,Friday,Fri,...,06/01/2023,01/06/2023,2023-01-06,2023-01-05,2023-01-06,2023-01-07,2023-01-01,2023-01-31,2023-01-01,2023-12-31
6,2023-01-07,2023,23,1,January,Jan,7,7th,Saturday,Sat,...,07/01/2023,01/07/2023,2023-01-07,2023-01-06,2023-01-07,2023-01-08,2023-01-01,2023-01-31,2023-01-01,2023-12-31
7,2023-01-08,2023,23,1,January,Jan,8,8th,Sunday,Sun,...,08/01/2023,01/08/2023,2023-01-08,2023-01-07,2023-01-08,2023-01-09,2023-01-01,2023-01-31,2023-01-01,2023-12-31
8,2023-01-09,2023,23,1,January,Jan,9,9th,Monday,Mon,...,09/01/2023,01/09/2023,2023-01-09,2023-01-08,2023-01-09,2023-01-10,2023-01-01,2023-01-31,2023-01-01,2023-12-31
9,2023-01-10,2023,23,1,January,Jan,10,10th,Tuesday,Tue,...,10/01/2023,01/10/2023,2023-01-10,2023-01-09,2023-01-10,2023-01-11,2023-01-01,2023-01-31,2023-01-01,2023-12-31
